# 🌾 Master End-to-End Walkthrough: Triangulated Rotation Advisor
**Team:** Gradient Descent Farmers · **Challenge:** NASA Space Apps — *Field Shift: Adapting Farms with NASA Data*

Este **Jupyter Notebook Maestro** documenta, ejecuta y valida de principio a fin toda la arquitectura implementada en el repositorio, sin depender de la interfaz gráfica web. Sirve como:
1. **Guía de arquitectura y código (con rutas exactas):** Explica qué hace cada archivo en `src/advisor/` y `web/src/`.
2. **Pipeline de ingesta y armonización de datos de la NASA:** Muestra cómo se consultan, cachean y limpian **NASA POWER**, **SRTM (30 m)**, **MODIS MOD13Q1 NDVI (250 m)** y **SoilGrids (250 m)**.
3. **Estandarización al contrato `SiteProfile`:** Verifica el esquema JSON universal (`src/advisor/schema.py`) y las banderas de disponibilidad (`ok`, `fallback`, `missing`).
4. **Cálculo de evidencia agronómica (`src/advisor/features.py`):** Evapotranspiración $ET_0$ (Hargreaves-Samani), corrección térmica por altitud (SRTM lapse-rate), probabilidad de heladas, rachas secas y detección satelital de **meses de suelo desnudo** ($	ext{NDVI} < 0.25$).
5. **Motor de Decisiones Triangulado + Optimización Multi-Objetivo (MOO) + Programación Dinámica (DP) (`src/advisor/engine.py` & `web/src/engine/`):**
   - **Lente 1 (Climate Twins):** Qué se cultiva hoy en regiones del mundo cuyo clima actual se parece al clima proyectado de tu parcela.
   - **Lente 2 (Opportunity Windows):** Ajuste del calendario del cultivo a la curva real de verdor satelital (MODIS NDVI) y cobertura de meses desnudos.
   - **Lente 3 (Resource Budget):** Simulación física ligera (balance hídrico FAO $K_y$, heladas, calor, nitrógeno y erosión) sobre 34 años históricos.
   - **Frontera de Pareto exacta** sobre 5 objetivos en conflicto.
   - **Programación Dinámica Estocástica (Bellman Backward Induction):** Política adaptativa año con año vs. rotación fija.
6. **Explicador con IA (Gemini 3.8 Flash + Guardrails anti-alucinación):** Cómo se construye el payload determinista y cómo funciona el filtro `ungroundedNumbers` antes de mostrar la explicación al agricultor.

---
### Mapa de Archivos del Proyecto
| Ruta en el repositorio | Rol dentro de la arquitectura |
|---|---|
| `src/advisor/schema.py` | Contrato JSON Schema (`SITE_PROFILE_SCHEMA` v0.1) que estandariza cualquier ubicación del planeta. |
| `src/advisor/pull.py` | Conectores HTTP con caché local (`data/advisor/cache/`) para NASA POWER, SRTM, SoilGrids y MODIS NDVI. |
| `src/advisor/features.py` | Fórmulas agronómicas puras: $ET_0$, corrección por altitud, heladas, sequías, meses desnudos de NDVI y AWC. |
| `src/advisor/build_profile.py` | Orquestador que une `pull.py` + `features.py` + `schema.py` y exporta `web/public/data/profiles/<cell_id>.json`. |
| `src/advisor/build_references.py` | Construye el catálogo global de 34 sitios agrícolas (`reference_sites.json`) para la Lente 1 (*Climate Twins*). |
| `src/advisor/engine.py` | Espejo en Python del motor de evaluación, las 3 lentes, la Frontera de Pareto y Programación Dinámica (DP). |
| `web/public/data/crops.json` | Catálogo estandarizado de 9 cultivos (ciclo, $K_y$, agua, fijación de N, cobertura, precios y costos base). |
| `web/public/data/rotation_rules.json` | Reglas agronómicas de penalización por plagas (misma familia botánica) y crédito de nitrógeno por leguminosas. |
| `web/src/engine/core.ts` & `evaluate.ts` | Motor en TypeScript que corre en el navegador en milisegundos al mover los sliders. |
| `web/src/ai/explain.ts` & `gemini.ts` | Integración con Firebase AI Logic (`gemini-3.8-flash`) y validador numérico anti-alucinaciones. |
| `web/src/components/*.tsx` | Componentes React del dashboard (`MapView`, `EvidencePanel`, `ControlsPanel`, `Results`). |

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Asegurar que la raíz del repositorio esté en sys.path
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "Notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.advisor import schema, features, pull, build_profile, engine
from src.data_ingestion import power

plt.rcParams["figure.figsize"] = (11, 4.2)
plt.rcParams["axes.grid"] = True
print(f"Repositorio raíz: {REPO_ROOT}")

---
## Paso 1: Ingesta de Datos y Caché Local (`src/advisor/pull.py`)

En lugar de descargar gigabytes en tiempo real cuando el agricultor hace clic, `src/advisor/pull.py` consulta APIs científicas ligeras y guarda cada respuesta cruda en `data/advisor/cache/` usando un hash SHA-1 de los parámetros:

1. **`pull.pull_elevation(lat, lon)`** → Consulta **SRTM 30 m** en el punto central y 4 vecinos ($\pm 0.003^\circ$) para calcular tanto la **altitud exacta ($\text{m}$)** como la **pendiente del terreno en grados ($\theta$)**.
2. **`pull.pull_power(lat, lon)`** → Descarga 34 años (1991–2024) de clima diario agroclimático (**NASA POWER**, comunidad `AG`): `T2M`, `T2M_MAX`, `T2M_MIN`, `PRECTOTCORR`, `RH2M`, `WS2M`, `ALLSKY_SFC_SW_DWN`.
3. **`pull.pull_modis_ndvi(lat, lon)`** → Descarga compuestos de 16 días de **MODIS MOD13Q1 (250 m)** vía el servicio de subconjuntos de ORNL DAAC.
4. **`pull.pull_soilgrids(lat, lon)`** → Consulta **ISRIC SoilGrids (250 m)** para obtener carbono orgánico (`soc`), arcilla (`clay`), arena (`sand`), `phh2o` y nitrógeno total (`nitrogen`) ponderados de 0 a 30 cm.

Veamos los archivos que ya fueron descargados en caché para nuestro sitio piloto en **Texcoco, Estado de México ($19.53^\circ\text{N}, -98.85^\circ\text{W}$)**:

In [ ]:
cache_files = sorted((REPO_ROOT / "data" / "advisor" / "cache").glob("*.json"))
for f in cache_files:
    print(f"• {f.name:32s}  ({f.stat().st_size / 1024:7.1f} KB)")

---
## Paso 2: Transformación Física y Evidencia Agronómica (`src/advisor/features.py`)

Aquí resolvemos uno de los problemas clave que encontró Isaac en su calibración de WOFOST:
- **Sesgo de altitud en NASA POWER:** La celda de ~50 km de NASA POWER en Texcoco tiene una elevación media de **$2,633\text{ m}$**, mientras que los campos agrícolas en el valle están a **$2,253\text{ m}$** ($380\text{ m}$ más abajo). Sin corrección, NASA POWER reporta noches artificialmente frías.
- **`features.correct_temperature(daily, grid_elevation_m, site_elevation_m)`** aplica el gradiente adiabático estándar ($0.0065\text{ }^\circ\text{C/m}$) usando la elevación SRTM de 30 m.
- **`features.hargreaves_et0(...)`** calcula la evapotranspiración de referencia diaria ($ET_0$, mm/día) a partir de la radiación extraterrestre FAO-56 ($R_a$) y el rango térmico diario.

Ejecutemos esta transformación directamente sobre la caché de NASA POWER + SRTM de Texcoco:

In [ ]:
LAT, LON = 19.53, -98.85
(elev_m, slope_deg), elev_status = pull.pull_elevation(LAT, LON)
power_payload, power_status = pull.pull_power(LAT, LON)

daily_raw = power.to_dataframe(power_payload)
grid_elev_m = power_payload["geometry"]["coordinates"][2]
daily_corr, temp_shift_c = features.correct_temperature(daily_raw, grid_elev_m, elev_m)

print(f"Elevación celda NASA POWER : {grid_elev_m:.1f} m")
print(f"Elevación parcela SRTM 30m : {elev_m:.1f} m (pendiente: {slope_deg:.2f}°)")
print(f"Corrección térmica aplicada: +{temp_shift_c:.2f} °C a T2M, T2M_MIN y T2M_MAX")
print(f"Días totales procesados    : {len(daily_corr):,} días ({daily_corr.index.min().date()} a {daily_corr.index.max().date()})")
daily_corr.tail(5)

---
## Paso 3: Estandarización al Contrato Universal `SiteProfile` (`src/advisor/schema.py` y `build_profile.py`)

Para que el motor de decisiones y la app web funcionen en **cualquier punto del planeta** (Texcoco, Iowa, Punjab o una parcela arbitraria), `build_profile.py` empaqueta todas las fuentes en un único esquema validado por `jsonschema` (`SITE_PROFILE_SCHEMA`).

Carguemos el archivo `web/public/data/profiles/texcoco_mx_19.53_-98.85.json` generado y verifiquemos su validez y sus banderas de disponibilidad:

In [ ]:
profile_path = REPO_ROOT / "web" / "public" / "data" / "profiles" / "texcoco_mx_19.53_-98.85.json"
profile = json.loads(profile_path.read_text())

# Validación estricta contra SITE_PROFILE_SCHEMA
schema.validate(profile)
print("✅ Esquema validado exitosamente contra SITE_PROFILE_SCHEMA v" + profile["schema_version"])
print("\nEstado de disponibilidad por capa (transparencia ante brechas de datos):")
avail_df = pd.DataFrame.from_dict(profile["availability"], orient="index")
display(avail_df)

### Visualización de la Evidencia Satelital en el `SiteProfile`
Con los arreglos de 12 meses dentro de `profile["climate_monthly"]` y `profile["vegetation"]`, podemos graficar exactamente la misma evidencia que muestra el panel `EvidencePanel.tsx` de la web app:
1. **Balance hídrico mensual:** Lluvia (mm) vs. Demanda evaporativa de la atmósfera ($ET_0$, mm) y temperatura media.
2. **Curva satelital de verdor (MODIS NDVI):** Identificación automática de **meses de suelo típicamente desnudo** ($\text{NDVI} < 0.25$), donde el suelo queda expuesto a erosión y pérdida de humedad.
3. **Riesgo de heladas y rachas secas:** Probabilidad diaria de helada por mes y duración máxima sin lluvia.

In [ ]:
months = engine.MONTHS
cm = profile["climate_monthly"]
veg = profile["vegetation"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# 1. Balance hídrico
ax = axes[0]
ax.bar(months, cm["rain_mm"], color="#1976d2", alpha=0.75, label="Lluvia (mm)")
ax.plot(months, cm["et0_mm"], color="#ef6c00", marker="o", linewidth=2, label="ET₀ ref. (mm)")
ax.set_title("Balance Hídrico Mensual (NASA POWER)")
ax.set_ylabel("mm / mes")
ax.legend(loc="upper left")

# 2. MODIS NDVI y meses desnudos
ax = axes[1]
ax.plot(months, veg["ndvi_monthly"], color="#2e7d32", marker="o", linewidth=2.5, label="NDVI medio (MODIS 250m)")
ax.plot(months, veg["ndvi_p10_monthly"], color="#c62828", linestyle="--", label="NDVI año malo (P10)")
ax.axhline(0.25, color="#78909c", linestyle=":", label="Umbral suelo desnudo (0.25)")
for m_idx in veg["bare_months"]:
    ax.axvspan(m_idx - 1.4, m_idx - 0.6, color="#ffccbc", alpha=0.45)
ax.set_title(f"Verdor Satelital y Meses Desnudos: {[months[m-1] for m in veg['bare_months']]}")
ax.set_ylim(0, 0.85)
ax.legend(loc="upper left")

# 3. Heladas y rachas secas
ax = axes[2]
frost_pct = [round((p or 0) * 100, 1) for p in cm["frost_prob"]]
ax.bar(months, frost_pct, color="#5c6bc0", alpha=0.75, label="Prob. helada (%)")
ax2 = ax.twinx()
ax2.plot(months, cm["dry_spell_days"], color="#d84315", marker="s", linewidth=2, label="Racha seca máx (días)")
ax.set_title("Riesgo de Helada (corregido SRTM) y Rachas Secas")
ax.set_ylabel("Probabilidad de helada (%)")
ax2.set_ylabel("Días secos consecutivos")
ax2.grid(False)

plt.tight_layout()
plt.show()

---
## Paso 4: Catálogo de Cultivos (`crops.json`) y Calendario Óptimo por Cultivo

Cargamos `web/public/data/crops.json` y `web/public/data/rotation_rules.json`. Para cada cultivo, `engine.build_crop_tables(crops, profile)` prueba todos los meses de siembra posibles a lo largo de los 34 años históricos de clima y elige la ventana que maximiza el factor de rendimiento (agua FAO $K_y$, temperatura óptima, penalización por olas de calor y supervivencia a heladas):

In [ ]:
crops_data = json.loads((REPO_ROOT / "web" / "public" / "data" / "crops.json").read_text())["crops"]
rules_data = json.loads((REPO_ROOT / "web" / "public" / "data" / "rotation_rules.json").read_text())

tables = engine.build_crop_tables(crops_data, profile)

summary_rows = []
for c in crops_data:
    t = tables[c["id"]]
    by_y = t["byYear"]
    summary_rows.append({
        "Cultivo": c["name"],
        "Tipo": c["type"],
        "Ciclo (meses)": c["cycle_months"],
        "Mes óptimo siembra": t["sowingMonthName"],
        "Rend. medio (t/ha)": round(float(np.mean([o["yieldT"] for o in by_y])), 2),
        "Ingreso neto medio ($/ha)": round(float(np.mean([o["netUsd"] for o in by_y])), 0),
        "Años con daño por helada (%)": round(t["frostDamageRate"] * 100, 1),
        "Años de pérdida severa (%)": round(t["failRate"] * 100, 1),
    })

pd.DataFrame(summary_rows)

---
## Paso 5: Evaluación Multi-Objetivo (MOO) y Frontera de Pareto Exacta (`src/advisor/engine.py`)

Ahora evaluamos **todas las secuencias válidas de rotación de 3 años** combinando las 3 lentes:
- **Objetivos optimizados simultáneamente:**
  1. `income`: Ingreso neto anual esperado ($\text{USD/ha/año}$), incluyendo el ahorro en fertilizante nitrogenado cuando un cereal sigue a una leguminosa.
  2. `badYear`: Ingreso neto promedio en el **peor 10% de los años climáticos (CVaR)** — mide resiliencia ante sequías/heladas.
  3. `soil`: Índice compuesto de salud del suelo ($0\text{–}1$): aporte de residuos orgánicos ($30\%$), fijación biológica de nitrógeno ($25\%$), cobertura viva ($20\%$) y ruptura de ciclos de plagas por alternancia de familias botánicas ($25\%$). 
  4. `water`: Demanda media de agua por temporada ($\text{mm}$, a minimizar).
  5. `erosion`: Índice de riesgo de erosión ($0\text{–}1$, a minimizar) basado en pendiente SRTM, días de lluvia extrema y meses descubiertos.
- **`engine.pareto_front_mask(df)`** identifica el conjunto exacto de planes **Pareto-óptimos** (no dominados).

In [ ]:
priorities = {"income": 0.25, "badYear": 0.25, "soil": 0.25, "water": 0.15, "erosion": 0.10}
df_rot, _ = engine.evaluate_all_rotations(profile, crops_data, rules_data, weights=priorities)

print(f"Rotaciones de 3 años evaluadas : {len(df_rot)}")
print(f"Rotaciones en la Frontera Pareto: {int(df_rot['is_pareto'].sum())}")

cols = ["cropNames", "is_pareto", "utility", "income", "badYear", "soil", "water", "erosion", "lens_windows"]
print("\nTop 10 rotaciones según las prioridades seleccionadas:")
display(df_rot[cols].head(10).round(2))

# Comparación directa contra el monocultivo tradicional (Maíz -> Maíz -> Maíz)
mono = df_rot[df_rot["crops"].apply(lambda x: x == ["maize", "maize", "maize"])]
print("\nLínea base (Monocultivo Maíz → Maíz → Maíz):")
display(mono[cols].round(2))

In [ ]:
# Gráfica de la Frontera de Pareto: Ingreso Neto vs. Salud del Suelo (coloreado por Uso de Agua)
pareto_df = df_rot[df_rot["is_pareto"]]
dom_df = df_rot[~df_rot["is_pareto"]]

fig, ax = plt.subplots(figsize=(10, 5.2))
ax.scatter(dom_df["soil"], dom_df["income"], c="#cfd8dc", s=28, alpha=0.6, label="Candidatas dominadas")
sc = ax.scatter(
    pareto_df["soil"], pareto_df["income"],
    c=pareto_df["water"], cmap="viridis_r", s=75, edgecolors="black", linewidth=0.6,
    label="Frontera de Pareto (no dominadas)"
)
# Destacar monocultivo de maíz
ax.scatter(mono["soil"], mono["income"], color="#d32f2f", marker="X", s=160, zorder=5, label="Actual: Maíz → Maíz → Maíz")

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Uso medio de agua (mm/temporada, menor es mejor)")
ax.set_xlabel("Índice de Salud del Suelo (0–1)")
ax.set_ylabel("Ingreso Neto Esperado (USD / ha / año)")
ax.set_title("Frontera de Pareto Multi-Objetivo — Texcoco, México")
ax.legend(loc="lower left")
plt.show()

---
## Paso 6: Programación Dinámica Estocástica (DP / MDP) — Política Adaptativa vs. Plan Fijo

Mientras que MOO elige una secuencia fija por adelantado, **`engine.solve_adaptive_dp(...)`** resuelve la ecuación de Bellman por inducción hacia atrás (*backward induction*):
$$V_t(s) = \max_{a \in \mathcal{A}} \left\{ R(s, a) + \gamma \, V_{t+1}(s') \right\}$$
- **Estado $s = (\text{cultivo previo}, \text{nivel de N en suelo})$:** Donde el nivel de nitrógeno es `high` después de una leguminosa/cobertura ($\ge 40\text{ kg N/ha}$ fijados) y `low` después de un cereal.
- **Acción $a$:** Qué cultivo sembrar en el siguiente ciclo.
- **Recompensa $R(s, a)$:** Combina ingreso esperado, protección en años malos (CVaR), bono por salud de suelo y penalización por repetir la misma especie o familia botánica.

Veamos la tabla de decisión óptima (Política $\pi^*(s)$) que aprende DP para Texcoco:

In [ ]:
dp_res = engine.solve_adaptive_dp(profile, crops_data, rules_data, tables, horizon=6, discount=0.95)
crop_names = {c["id"]: c["name"] for c in crops_data}

policy_rows = []
for (prev_id, n_bin), next_id in dp_res["policy"].items():
    policy_rows.append({
        "Cultivo recién cosechado": crop_names[prev_id],
        "Estado de Nitrógeno en suelo": "Alto (post-leguminosa)" if n_bin == "high" else "Bajo (agotado)",
        "Acción óptima DP (Qué sembrar ahora)": crop_names[next_id],
        "Valor acumulado esperado V(s)": round(dp_res["value_function"][(prev_id, n_bin)], 1),
    })

pd.DataFrame(policy_rows)

---
## Paso 7: Payload para Gemini 3.8 Flash y Guardrail Anti-Alucinaciones (`web/src/ai/explain.ts`)

En la aplicación web, cuando el agricultor pulsa **"Explain to Farmer"**:
1. `explainerPayload(...)` empaqueta **únicamente** los números calculados en la tarjeta seleccionada y sus etiquetas de evidencia (`observed-satellite`, `observed-practice`, `model-estimate`, `rule-of-thumb`).
2. Se envía a **Firebase AI Logic (`gemini-3.8-flash`)** con `responseSchema` estricto.
3. Al recibir el JSON, `ungroundedNumbers(explanation, payload)` extrae todas las cifras de $\ge 2$ dígitos del texto generado y verifica que existan en el JSON de entrada. Si el modelo inventa cualquier cifra, la respuesta se descarta automáticamente y se muestra el explicador determinista.

Aquí podemos inspeccionar el payload exacto que recibe Gemini para la mejor rotación en Texcoco:

In [ ]:
best_row = df_rot.iloc[0]
sample_gemini_payload = {
    "strategy": "Best for your priorities",
    "rotation": best_row["cropNames"].split(" → "),
    "metrics": {
        "avgNetIncomeUsdPerHaYear": round(float(best_row["income"])),
        "worstTenPercentYearsUsdPerHaYear": round(float(best_row["badYear"])),
        "soilHealthIndex0to1": round(float(best_row["soil"]), 2),
        "waterUseMmPerSeason": round(float(best_row["water"])),
        "erosionRiskIndex0to1": round(float(best_row["erosion"]), 2),
        "pestBreakScore0to1": round(float(best_row["pestBreak"]), 2),
    },
    "evidence_lenses": {
        "budget_percentile": round(float(best_row["pct_budget"]) * 100, 1),
        "ndvi_windows_percentile": round(float(best_row["pct_windows"]) * 100, 1),
    }
}
print(json.dumps(sample_gemini_payload, indent=2, ensure_ascii=False))